In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import colormaps as cm
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import json

import clustergraph.clustergraph as cg
from clustergraph.utils import get_clusters_from_scikit
from clustergraph.plot_graph import draw_graph

def corrupt_labels(labels, corruption_rate, seed=42):
    """
    Randomly reassign a fraction `corruption_rate` of points to a
    different cluster. Each corrupted point is assigned uniformly at
    random to one of the other clusters (never its own).
    """
    rng = np.random.default_rng(seed)
    labels_new = labels.copy()
    n = len(labels_new)
    n_corrupt = int(n * corruption_rate)
    unique_clusters = np.unique(labels)
    corrupt_idx = rng.choice(n, size=n_corrupt, replace=False)
    for idx in corrupt_idx:
        current = labels[idx]
        other_clusters = unique_clusters[unique_clusters != current]
        labels_new[idx] = rng.choice(other_clusters)
    return labels_new

# (1) Concentric circles

In [ ]:
%%time
data = pd.read_csv("data/noisy_circles.csv", sep=",", header=None)
X = data.to_numpy()
print(X.shape)

k_knn_graph = [i for i in range(5, 21)]
k_kmeans = [i for i in range(15, 25)]

seed = 42
results = []

for kcurrent_knn in k_knn_graph:
    for kcurrent_kmeans in k_kmeans:
        
        model_KM = KMeans(kcurrent_kmeans, random_state=seed, n_init=10)
        prediction_KM = model_KM.fit_predict(X)

        cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(prediction_KM), X=X, metric_clusters="average"
        )

        cluster_g.color_graph(
            node_color_labels=prediction_KM,
            node_palette=cm.get_cmap("tab20"),
        )

        metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=kcurrent_knn, score=True)
        
        print(f"--- k_knn={kcurrent_knn} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
        
        results.append({
            "k_knn": kcurrent_knn,
            "k_kmeans": kcurrent_kmeans,
            "md_min": min(md),
            "md_max": max(md),
            "seed_kmeans": seed,
            "metrics": "average/euclidean",
            "kmeans_n_init": 10
        })

# Build summary DataFrame
results_df = pd.DataFrame(results)

# Pivot tables for min and max
pivot_min = results_df.pivot(index="k_knn", columns="k_kmeans", values="md_min")
pivot_max = results_df.pivot(index="k_knn", columns="k_kmeans", values="md_max")

print("\n===== Metric Distortion MIN (rows=k_knn, cols=k_kmeans) =====")
display(pivot_min.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

print("\n===== Metric Distortion MAX (rows=k_knn, cols=k_kmeans) =====")
display(pivot_max.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

# Full flat table
print("\n===== Full Results Table =====")
display(results_df.sort_values(["k_knn", "k_kmeans"]).reset_index(drop=True))

In [ ]:
import json

# Save full results to JSON
with open("ClusterGraph_results_stability/sensitivity_analysis_concentric_circles.json", "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to ClusterGraph_results_stability/sensitivity_analysis_concentric_circles.json")

### Robutness to noisy clustering Concentric circles

In [ ]:
data = pd.read_csv("data/noisy_circles.csv", sep=",", header=None)
X = data.to_numpy()
print(X.shape)

In [ ]:
%%time

k_kmeans = [16, 18, 20]
k_knn = 10
kmeans_seed=42
nb_seeds=10
corruption_rates = [0.02, 0.05, 0.07, 0.10, 0.12, 0.15]

results_noise_concentric_circles=[]
for kcurrent_kmeans in k_kmeans:
    model_KM = KMeans(kcurrent_kmeans, random_state=kmeans_seed, n_init=10)
    prediction_KM = model_KM.fit_predict(X)
    
    for cor_rate in corruption_rates:
        md_min_avg=0
        md_max_avg=0

        for seed in range(nb_seeds):
            corrupted_labels = corrupt_labels(prediction_KM, cor_rate, seed=seed)
            
            # CG part 
            cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(corrupted_labels), X=X, metric_clusters="average"
            )

            metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=k_knn, score=True)

            print(f"--- k_knn={k_knn} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
            md_min_avg+=min(md)
            md_max_avg+=max(md)

        results_noise_concentric_circles.append({
            "nb_seed_per_rate":nb_seeds,
            "corrupted_rate":cor_rate,
            "k_knn": k_knn,
            "k_kmeans": kcurrent_kmeans,
            "md_min_avg": md_min_avg/nb_seeds,
            "md_max_avg": md_max_avg/nb_seeds,
            "seed_kmeans": kmeans_seed,
            "metrics": "average/euclidean",
            "kmeans_n_init": 10
        })
        
# Save full results to JSON
with open("ClusterGraph_results_stability/noise_clustering_robustness_analysis_concentric_circles.json", "w") as f:
    json.dump(results_noise_concentric_circles, f, indent=4)

print("Results saved to noise_clustering_robustness_analysis_concentric_circles.json")

# (2) Mice Protein Expression

In [ ]:
df = pd.read_csv("data/mice_protein_no_NaN.csv")
X = df.iloc[:, :-1].to_numpy()
scaler = StandardScaler()
X = scaler.fit_transform(X)
pca = PCA(0.95)
X_pca = pca.fit_transform(X)

k_knn_graph_mice_protein = [i for i in range(10,31)] #10 à 30; one connected component
k_kmeans_mice_protein = [i for i in range(10,21)]
seed=42

In [ ]:
results_mice_protein = []

for kcurrent_knn in k_knn_graph_mice_protein:
    for kcurrent_kmeans in k_kmeans_mice_protein:
        model_KM = KMeans(kcurrent_kmeans, random_state=seed, n_init=10)
        prediction_KM = model_KM.fit_predict(X_pca)
        cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(prediction_KM), X=X_pca, metric_clusters="average"
        )
        cluster_g.color_graph(
            node_color_labels=prediction_KM,
            node_palette=cm.get_cmap("tab20"),
        )
        metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=kcurrent_knn, score=True)
        
        print(f"--- k_knn={kcurrent_knn} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
        
        results_mice_protein.append({
            "k_knn": kcurrent_knn,
            "k_kmeans": kcurrent_kmeans,
            "md_min": min(md),
            "md_max": max(md),
            "seed_kmeans": seed,
            "metrics": "average/euclidean",
            "kmeans_n_init": 10
        })

# Pivot tables
results_mice_df = pd.DataFrame(results_mice_protein)

pivot_min = results_mice_df.pivot(index="k_knn", columns="k_kmeans", values="md_min")
pivot_max = results_mice_df.pivot(index="k_knn", columns="k_kmeans", values="md_max")

print("\n===== Metric Distortion MIN (rows=k_knn, cols=k_kmeans) =====")
display(pivot_min.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

print("\n===== Metric Distortion MAX (rows=k_knn, cols=k_kmeans) =====")
display(pivot_max.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

print("\n===== Full Results Table =====")
display(results_mice_df.sort_values(["k_knn", "k_kmeans"]).reset_index(drop=True))

# Save to JSON
with open("ClusterGraph_results_stability/sensitivity_analysis_mice_protein.json", "w") as f:
    json.dump(results_mice_protein, f, indent=4)

print("Results saved to sensitivity_analysis_mice_protein.json")

### Noisy clustering mice protein

In [ ]:
df_mice_protein = pd.read_csv("data/mice_protein_no_NaN.csv")
X_mice_protein = df_mice_protein.iloc[:, :-1].to_numpy()
scaler = StandardScaler()
X_mice_protein = scaler.fit_transform(X_mice_protein)
pca = PCA(0.95)
X_pca_mice_protein = pca.fit_transform(X_mice_protein)

In [ ]:
%%time

k_kmeans_mice_protein = [14, 16, 18]
k_knn_mice_protein = 15
kmeans_seed=42
nb_seeds=10
corruption_rates = [0.02, 0.05, 0.07, 0.10, 0.12, 0.15]

results_noise_mice_protein=[]
for kcurrent_kmeans in k_kmeans_mice_protein:
    model_KM = KMeans(kcurrent_kmeans, random_state=kmeans_seed, n_init=10)
    prediction_KM = model_KM.fit_predict(X_pca_mice_protein)
    
    for cor_rate in corruption_rates:
        md_min_avg=0
        md_max_avg=0

        for seed in range(nb_seeds):
            corrupted_labels = corrupt_labels(prediction_KM, cor_rate, seed=seed)
            
            # CG part 
            cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(corrupted_labels), X=X_pca_mice_protein, metric_clusters="average"
            )

            metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=k_knn_mice_protein, score=True)

            print(f"--- k_knn={k_knn_mice_protein} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
            md_min_avg+=min(md)
            md_max_avg+=max(md)

        results_noise_mice_protein.append({
            "nb_seed_per_rate":nb_seeds,
            "corrupted_rate":cor_rate,
            "k_knn": k_knn_mice_protein,
            "k_kmeans": kcurrent_kmeans,
            "md_min_avg": md_min_avg/nb_seeds,
            "md_max_avg": md_max_avg/nb_seeds,
            "seed_kmeans": kmeans_seed,
            "metrics": "average/euclidean",
            "kmeans_n_init": 10
        })
        
# Save full results to JSON
with open("ClusterGraph_results_stability/noise_clustering_robustness_analysis_mice_protein.json", "w") as f:
    json.dump(results_noise_mice_protein, f, indent=4)

print("Results saved to noise_clustering_robustness_analysis_mice_protein.json")

# (3) Diabete dataset

In [ ]:
%%time
data = pd.read_csv("data/diabete.csv", sep=",")
X = data.iloc[:, :-1].to_numpy()
print(X.shape)

k_knn_graph_diabete = [i for i in range(5, 25)]
k_kmeans_diabete = [i for i in range(5, 20)]

seed = 42
results_diabete = []

for kcurrent_knn in k_knn_graph_diabete:
    for kcurrent_kmeans in k_kmeans_diabete:
        
        model_KM = KMeans(kcurrent_kmeans, random_state=seed, n_init=10)
        prediction_KM = model_KM.fit_predict(X)
        cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(prediction_KM), X=X, metric_clusters="average"
        )
        cluster_g.color_graph(
            node_color_labels=prediction_KM,
            node_palette=cm.get_cmap("tab20"),
        )
        metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=kcurrent_knn, score=True)
        
        print(f"--- k_knn={kcurrent_knn} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
        
        results_diabete.append({
            "k_knn": kcurrent_knn,
            "k_kmeans": kcurrent_kmeans,
            "md_min": min(md),
            "md_max": max(md),
            "seed_kmeans": seed,
            "metrics": "average/euclidean",
            "kmeans_n_init": 10
        })

# Pivot tables
results_diabete_df = pd.DataFrame(results_diabete)

pivot_min = results_diabete_df.pivot(index="k_knn", columns="k_kmeans", values="md_min")
pivot_max = results_diabete_df.pivot(index="k_knn", columns="k_kmeans", values="md_max")

print("\n===== Metric Distortion MIN (rows=k_knn, cols=k_kmeans) =====")
display(pivot_min.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

print("\n===== Metric Distortion MAX (rows=k_knn, cols=k_kmeans) =====")
display(pivot_max.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

print("\n===== Full Results Table =====")
display(results_diabete_df.sort_values(["k_knn", "k_kmeans"]).reset_index(drop=True))

# Save to JSON
with open("ClusterGraph_results_stability/sensitivity_analysis_diabete.json", "w") as f:
    json.dump(results_diabete, f, indent=4)

print("Results saved to sensitivity_analysis_diabete.json")

### Robustness to noisy clustering Diabete

In [ ]:
data_diabete = pd.read_csv("data/diabete.csv", sep=",")
X_diabete = data_diabete.iloc[:, :-1].to_numpy()
print(X_diabete.shape)

In [ ]:
%%time

k_kmeans_diabete = [7, 10, 13]
k_knn_diabete = 10
kmeans_seed=42
nb_seeds=10
corruption_rates = [0.02, 0.05, 0.07, 0.10, 0.12, 0.15]

results_noise_diabete=[]
for kcurrent_kmeans in k_kmeans_diabete:
    model_KM = KMeans(kcurrent_kmeans, random_state=kmeans_seed, n_init=10)
    prediction_KM = model_KM.fit_predict(X_diabete)
    
    for cor_rate in corruption_rates:
        md_min_avg=0
        md_max_avg=0

        for seed in range(nb_seeds):
            corrupted_labels = corrupt_labels(prediction_KM, cor_rate, seed=seed)
            
            # CG part 
            cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(corrupted_labels), X=X_diabete, metric_clusters="average"
            )

            metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=k_knn_diabete, score=True)

            print(f"--- k_knn={k_knn_diabete} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
            md_min_avg+=min(md)
            md_max_avg+=max(md)

        results_noise_diabete.append({
            "nb_seed_per_rate":nb_seeds,
            "corrupted_rate":cor_rate,
            "k_knn": k_knn_diabete,
            "k_kmeans": kcurrent_kmeans,
            "md_min_avg": md_min_avg/nb_seeds,
            "md_max_avg": md_max_avg/nb_seeds,
            "seed_kmeans": kmeans_seed,
            "metrics": "average/euclidean",
            "kmeans_n_init": 10
        })
        
# Save full results to JSON
with open("ClusterGraph_results_stability/noise_clustering_robustness_analysis_diabete.json", "w") as f:
    json.dump(results_noise_diabete, f, indent=4)

print("Results saved to noise_clustering_robustness_analysis_diabete.json")

# (4) Lung cancer

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

counts = pd.read_csv("data/sc_10x_5cl.count.csv")
metadata = pd.read_csv("data/sc_10x_5cl.metadata.csv")
## these are the 5 cell lines
print( metadata.cell_line_demuxlet.value_counts() )

# Create an AnnData object
adata = ad.AnnData(counts.T)  # need to transpose because I want the genes to be columns
adata.obs = metadata
print("Dimensions data : ", adata.X.shape)

# Normalize counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Logarithmize the data
sc.pp.log1p(adata)

# Identify highly variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# Keep only those genes
adata = adata[:, adata.var.highly_variable]

# Check the dataset after filtering
print(adata)

# Run PCA
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")

# Visualize the PCA
sc.pl.pca_variance_ratio(adata, log=True)

# Compute the neighborhood graph
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

n_pcs = 40
adata.obsm["X_pca"][:, :n_pcs]

n_pcs = 40
X_lung_cancer = adata.obsm["X_pca"][:, :n_pcs]
pred_lung_cancer = np.array(metadata["cell_line_demuxlet"])

In [ ]:
with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "w") as f:
    json.dump([], f)
print("JSON initialized")

In [ ]:
%%time
k_knn_chunk        = [10, 11, 12, 13]
k_kmeans_lung_cancer = [i for i in range(10, 16)]
seed = 42

# Load existing results
with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "r") as f:
    results_lung_cancer = json.load(f)

for kcurrent_knn in k_knn_chunk:
    for kcurrent_kmeans in k_kmeans_lung_cancer:
        model_KM = KMeans(kcurrent_kmeans, random_state=seed, n_init=10)
        prediction_KM = model_KM.fit_predict(X_lung_cancer)
        cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(prediction_KM), X=X_lung_cancer, metric_clusters="average"
        )
        cluster_g.color_graph(
            node_color_labels=prediction_KM,
            node_palette=cm.get_cmap("tab20"),
        )
        metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=kcurrent_knn, score=True)
        print(f"--- k_knn={kcurrent_knn} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
        results_lung_cancer.append({
            "k_knn":         kcurrent_knn,
            "k_kmeans":      kcurrent_kmeans,
            "md_min":        min(md),
            "md_max":        max(md),
            "seed_kmeans":   seed,
            "metrics":       "average/euclidean",
            "kmeans_n_init": 10
        })

# Save after this chunk
with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "w") as f:
    json.dump(results_lung_cancer, f, indent=4)
print(f"Saved {len(results_lung_cancer)} entries so far")

In [ ]:
%%time
k_knn_chunk        = [14, 15, 16, 17]
k_kmeans_lung_cancer = [i for i in range(10, 16)]
seed = 42

with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "r") as f:
    results_lung_cancer = json.load(f)

for kcurrent_knn in k_knn_chunk:
    for kcurrent_kmeans in k_kmeans_lung_cancer:
        model_KM = KMeans(kcurrent_kmeans, random_state=seed, n_init=10)
        prediction_KM = model_KM.fit_predict(X_lung_cancer)
        cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(prediction_KM), X=X_lung_cancer, metric_clusters="average"
        )
        cluster_g.color_graph(
            node_color_labels=prediction_KM,
            node_palette=cm.get_cmap("tab20"),
        )
        metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=kcurrent_knn, score=True)
        print(f"--- k_knn={kcurrent_knn} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
        results_lung_cancer.append({
            "k_knn":         kcurrent_knn,
            "k_kmeans":      kcurrent_kmeans,
            "md_min":        min(md),
            "md_max":        max(md),
            "seed_kmeans":   seed,
            "metrics":       "average/euclidean",
            "kmeans_n_init": 10
        })

with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "w") as f:
    json.dump(results_lung_cancer, f, indent=4)
print(f"Saved {len(results_lung_cancer)} entries so far")

In [ ]:
%%time
k_knn_chunk        = [18, 19, 20]
k_kmeans_lung_cancer = [i for i in range(10, 16)]
seed = 42

with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "r") as f:
    results_lung_cancer = json.load(f)

for kcurrent_knn in k_knn_chunk:
    for kcurrent_kmeans in k_kmeans_lung_cancer:
        model_KM = KMeans(kcurrent_kmeans, random_state=seed, n_init=10)
        prediction_KM = model_KM.fit_predict(X_lung_cancer)
        cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(prediction_KM), X=X_lung_cancer, metric_clusters="average"
        )
        cluster_g.color_graph(
            node_color_labels=prediction_KM,
            node_palette=cm.get_cmap("tab20"),
        )
        metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=kcurrent_knn, score=True)
        print(f"--- k_knn={kcurrent_knn} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
        results_lung_cancer.append({
            "k_knn":         kcurrent_knn,
            "k_kmeans":      kcurrent_kmeans,
            "md_min":        min(md),
            "md_max":        max(md),
            "seed_kmeans":   seed,
            "metrics":       "average/euclidean",
            "kmeans_n_init": 10
        })

with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "w") as f:
    json.dump(results_lung_cancer, f, indent=4)
print(f"Saved {len(results_lung_cancer)} entries so far")

In [ ]:
with open("ClusterGraph_results_stability/sensitivity_analysis_lung_cancer.json", "r") as f:
    results_lung_cancer = json.load(f)

results_lung_cancer_df = pd.DataFrame(results_lung_cancer)

pivot_min_lc = results_lung_cancer_df.pivot(index="k_knn", columns="k_kmeans", values="md_min")
pivot_max_lc = results_lung_cancer_df.pivot(index="k_knn", columns="k_kmeans", values="md_max")

print("\n===== Metric Distortion MIN (rows=k_knn, cols=k_kmeans) =====")
display(pivot_min_lc.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

print("\n===== Metric Distortion MAX (rows=k_knn, cols=k_kmeans) =====")
display(pivot_max_lc.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

print("\n===== Full Results Table =====")
display(results_lung_cancer_df.sort_values(["k_knn", "k_kmeans"]).reset_index(drop=True))

### Robustness to noisy clustering Lung Cancer

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

counts = pd.read_csv("data/sc_10x_5cl.count.csv")
metadata = pd.read_csv("data/sc_10x_5cl.metadata.csv")
## these are the 5 cell lines
print( metadata.cell_line_demuxlet.value_counts() )

# Create an AnnData object
adata = ad.AnnData(counts.T)  # need to transpose because I want the genes to be columns
adata.obs = metadata
print("Dimensions data : ", adata.X.shape)

# Normalize counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Logarithmize the data
sc.pp.log1p(adata)

# Identify highly variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# Keep only those genes
adata = adata[:, adata.var.highly_variable]

# Check the dataset after filtering
print(adata)

# Run PCA
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")

# Visualize the PCA
sc.pl.pca_variance_ratio(adata, log=True)

# Compute the neighborhood graph
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

n_pcs = 40
adata.obsm["X_pca"][:, :n_pcs]

n_pcs = 40
X_lung_cancer = adata.obsm["X_pca"][:, :n_pcs]
pred_lung_cancer = np.array(metadata["cell_line_demuxlet"])

In [ ]:
%%time

k_kmeans_lung_cancer = [11, 13, 15]
k_knn_lung_cancer = 14
kmeans_seed=42
nb_seeds=5
corruption_rates = [0.02, 0.05, 0.07, 0.10, 0.12]

results_noise_lung_cancer=[]
for kcurrent_kmeans in k_kmeans_lung_cancer:
    model_KM = KMeans(kcurrent_kmeans, random_state=kmeans_seed, n_init=10)
    prediction_KM = model_KM.fit_predict(X_lung_cancer)
    
    for cor_rate in corruption_rates:
        md_min_avg=0
        md_max_avg=0

        for seed in range(nb_seeds):
            corrupted_labels = corrupt_labels(prediction_KM, cor_rate, seed=seed)
            
            # CG part 
            cluster_g = cg.ClusterGraph(
            clusters=get_clusters_from_scikit(corrupted_labels), X=X_lung_cancer, metric_clusters="average"
            )

            metric_distortion_graph, md = cluster_g.prune_distortion(knn_g=k_knn_lung_cancer, score=True)

            print(f"--- k_knn={k_knn_lung_cancer} | k_kmeans={kcurrent_kmeans} --- Min: {min(md):.4f} | Max: {max(md):.4f}")
            md_min_avg+=min(md)
            md_max_avg+=max(md)

        results_noise_lung_cancer.append({
            "nb_seed_per_rate":nb_seeds,
            "corrupted_rate":cor_rate,
            "k_knn": k_knn_lung_cancer,
            "k_kmeans": kcurrent_kmeans,
            "md_min_avg": md_min_avg/nb_seeds,
            "md_max_avg": md_max_avg/nb_seeds,
            "seed_kmeans": kmeans_seed,
            "metrics": "average/euclidean",
            "kmeans_n_init": 10
        })
        
# Save full results to JSON
with open("ClusterGraph_results_stability/noise_clustering_robustness_analysis_lung_cancer.json", "w") as f:
    json.dump(results_noise_lung_cancer, f, indent=4)

print("Results saved to noise_clustering_robustness_analysis_lung_cancer.json")